In [ ]:
from __future__ import annotations

import traceback
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import numpy as np
import pandas as pd
import nd2  # replaces tifffile

from cellpose import models
from scipy.ndimage import gaussian_laplace
from scipy import ndimage as ndi
from scipy.stats import norm

from skimage import filters, morphology
from skimage.filters import threshold_otsu
from skimage.measure import regionprops_table
from skimage.segmentation import find_boundaries

from PIL import Image
from skimage.draw import rectangle_perimeter

def _imsave(path: Path, arr: np.ndarray):
    """Save a uint8 array as PNG using Pillow (avoids imageio/tifffile backend issues)."""
    Image.fromarray(arr).save(path)


# ----------------------------
# Helpers
# ----------------------------
def get_channel_indices(row: pd.Series, n_channels: int = 4) -> Dict[str, int]:
    """Map channel name -> channel index based on columns ch0..ch{n-1}."""
    channel_map: Dict[str, int] = {}
    for i in range(n_channels):
        name = row.get(f"ch{i}")
        if isinstance(name, str) and name.strip():
            channel_map[name.strip()] = i
    return channel_map

def build_img_lookup(img_dir: Path, pattern: str = "*.nd2") -> Dict[str, Path]:
    imgs = sorted(img_dir.glob(pattern))
    return {p.name: p for p in imgs}

def add_image_paths(samplesheet: pd.DataFrame, img_lookup: Dict[str, Path]) -> pd.DataFrame:
    out = samplesheet.copy()
    out["image_path"] = out["filename"].map(lambda fn: img_lookup.get(fn))
    return out

def load_nd2_as_czyx(img_path: Path) -> np.ndarray:
    """
    Open an ND2 file and return a (C, H, W) array via max projection over Z.

    nd2.imread returns an array whose axis order is reported by the file's
    dimension labels (accessible via nd2.ND2File).  We normalise to (C, Z, H, W)
    before projecting so the rest of the pipeline receives the same (C, H, W)
    shape it previously got from tifffile.

    Supported axis orderings emitted by the nd2 package:
      CZYX, ZCYX, TCZYX (T is dropped by taking index 0), and the degenerate
      CYX / ZYX cases (no-Z or no-C, handled gracefully).
    """
    with nd2.ND2File(img_path) as f:
        # axes string, e.g. 'CZYX', 'ZCYX', 'TCZYX' …
        axes: str = "".join(f.sizes.keys()).upper()
        arr: np.ndarray = f.asarray()          # numpy array, dtype preserved

    # --- drop T if present (take first time-point) ---
    if "T" in axes:
        t_idx = axes.index("T")
        arr = arr.take(0, axis=t_idx)
        axes = axes.replace("T", "")

    # arr is now one of: CZYX, ZCYX, CYX, ZYX, YX …
    # Normalise to (C, Z, Y, X) or (C, Y, X) so we can max-project cleanly.

    if axes == "CZYX":
        pass                                   # already (C, Z, Y, X)
    elif axes == "ZCYX":
        arr = np.moveaxis(arr, 0, 1)           # → (C, Z, Y, X)
        axes = "CZYX"
    elif axes == "ZYX":
        arr = arr[np.newaxis]                  # → (1, Z, Y, X)  — single channel
        axes = "CZYX"
    elif axes == "CYX":
        # no Z dimension — nothing to project, just return
        return arr                             # (C, H, W)
    elif axes == "YX":
        return arr[np.newaxis]                 # (1, H, W)
    else:
        raise ValueError(
            f"Unexpected axis order '{axes}' in {img_path.name}. "
            "Please extend load_nd2_as_czyx() to handle it."
        )

    # Max-project over Z (axis 1) → (C, H, W)
    max_proj: np.ndarray = arr.max(axis=1)
    return max_proj


def to_uint8(img: np.ndarray, p_lo: float = 1.0, p_hi: float = 99.5) -> np.ndarray:
    """Percentile-stretch to uint8 for visualization."""
    x = img.astype(np.float32)
    lo, hi = np.percentile(x, [p_lo, p_hi])
    if hi <= lo:
        return np.zeros_like(x, dtype=np.uint8)
    x = np.clip((x - lo) / (hi - lo), 0, 1)
    return (255 * x).astype(np.uint8)

def find_edge_touching_labels(label_image: np.ndarray, margin: int = 0) -> set:
    h, w = label_image.shape
    labels = set()
    if margin <= 0:
        labels |= set(np.unique(label_image[0, :]))
        labels |= set(np.unique(label_image[h-1, :]))
        labels |= set(np.unique(label_image[:, 0]))
        labels |= set(np.unique(label_image[:, w-1]))
    else:
        labels |= set(np.unique(label_image[:margin, :]))
        labels |= set(np.unique(label_image[h-margin:, :]))
        labels |= set(np.unique(label_image[:, :margin]))
        labels |= set(np.unique(label_image[:, w-margin:]))
    labels.discard(0)
    return labels

def compute_edge_flags_bbox(cell_masks: np.ndarray, margin: int = 0) -> Dict[int, bool]:
    h, w = cell_masks.shape
    props = regionprops_table(cell_masks, properties=("label", "bbox"))
    if not props or len(props.get("label", [])) == 0:
        return {}

    labels = np.asarray(props["label"], dtype=int)
    ymin = np.asarray(props["bbox-0"], dtype=int)
    xmin = np.asarray(props["bbox-1"], dtype=int)
    ymax = np.asarray(props["bbox-2"], dtype=int)
    xmax = np.asarray(props["bbox-3"], dtype=int)

    flags: Dict[int, bool] = {}
    for lab, y0, x0, y1, x1 in zip(labels, ymin, xmin, ymax, xmax):
        touches = (y0 <= margin) or (x0 <= margin) or (y1 >= (h - margin)) or (x1 >= (w - margin))
        flags[int(lab)] = bool(touches)
    return flags


def compute_nuc_overlap_and_area_ratio(
    cell_masks: np.ndarray,
    nuc_masks: np.ndarray,
) -> Tuple[Dict[int, float], Dict[int, float]]:
    flat_cells = cell_masks.ravel()
    flat_nucs = nuc_masks.ravel()
    max_cell = int(flat_cells.max())
    max_nuc = int(flat_nucs.max())

    cell_area = np.bincount(flat_cells, minlength=max_cell + 1)
    nuc_area = np.bincount(flat_nucs, minlength=max_nuc + 1)

    mask_both = (flat_cells > 0) & (flat_nucs > 0)
    if mask_both.any():
        overlap_counts = np.bincount(flat_cells[mask_both], minlength=max_cell + 1)
    else:
        overlap_counts = np.zeros(max_cell + 1, dtype=int)

    overlap_pairs_cells = flat_cells[mask_both]
    overlap_pairs_nucs = flat_nucs[mask_both]

    overlap_frac = {}
    area_ratio = {}

    if overlap_pairs_cells.size == 0:
        for cid in np.unique(flat_cells):
            if cid == 0:
                continue
            a = int(cell_area[cid])
            overlap_frac[int(cid)] = 0.0
            area_ratio[int(cid)] = np.inf
        return overlap_frac, area_ratio

    cell_to_nucs = {}
    for c, n in zip(overlap_pairs_cells, overlap_pairs_nucs):
        cell_to_nucs.setdefault(int(c), set()).add(int(n))

    unique_cells = np.unique(flat_cells)
    unique_cells = unique_cells[unique_cells > 0]
    for cid in unique_cells:
        cid = int(cid)
        c_area = int(cell_area[cid]) if cid <= max_cell else 0
        ov = int(overlap_counts[cid]) if cid <= max_cell else 0
        overlap_frac[cid] = float(ov) / float(c_area) if c_area > 0 else 0.0

        nucs = cell_to_nucs.get(cid, set())
        if not nucs:
            area_ratio[cid] = np.inf
        else:
            nuc_areas = [int(nuc_area[nl]) for nl in nucs if nl <= max_nuc]
            largest = max(nuc_areas) if nuc_areas else 0
            area_ratio[cid] = float(c_area) / float(largest) if largest > 0 else np.inf

    return overlap_frac, area_ratio

def square_crop_coords(ymin, xmin, ymax, xmax, H, W, pad: int = 8):
    ymin, xmin, ymax, xmax = int(ymin), int(xmin), int(ymax), int(xmax)
    ymin = max(0, ymin - pad)
    xmin = max(0, xmin - pad)
    ymax = min(H, ymax + pad)
    xmax = min(W, xmax + pad)

    h = ymax - ymin
    w = xmax - xmin
    side = max(h, w)

    cy = (ymin + ymax) // 2
    cx = (xmin + xmax) // 2

    y0 = max(0, cy - side // 2)
    x0 = max(0, cx - side // 2)
    y1 = min(H, y0 + side)
    x1 = min(W, x0 + side)

    y0 = max(0, y1 - side)
    x0 = max(0, x1 - side)

    return y0, x0, y1, x1
 

def make_merged_rgb(dapi_u8, Phalloidin_u8, lamp1_u8, prab_u8):
    """
    Pseudocolor mapping:
      DAPI -> blue
      Phalloidin -> green
      LAMP1 -> red
      pRAB10 -> magenta (red + blue)
    """
    H, W = dapi_u8.shape
    rgb = np.zeros((H, W, 3), dtype=np.uint8)

    rgb[..., 0] = np.maximum(rgb[..., 0], lamp1_u8)
    rgb[..., 0] = np.maximum(rgb[..., 0], prab_u8)
    rgb[..., 1] = np.maximum(rgb[..., 1], Phalloidin_u8)
    rgb[..., 2] = np.maximum(rgb[..., 2], dapi_u8)
    rgb[..., 2] = np.maximum(rgb[..., 2], prab_u8)

    return rgb

def safe_merge_on_id(left: pd.DataFrame, right: pd.DataFrame, on: str = "cell_id", keep_on_left: bool = True) -> pd.DataFrame:
    if right is None or right.empty:
        return left.copy()

    left_cols = set(left.columns)
    right_cols = [c for c in right.columns if c != on]

    to_take = [c for c in right_cols if c not in left_cols]
    if not to_take:
        return left.copy()

    right_sel = right[[on] + to_take].copy()
    merged = left.merge(right_sel, on=on, how="left", validate="one_to_one")
    return merged

def cellpose_outline_from_labels(
    cell_masks: np.ndarray,
    cid: int,
    thickness: int = 2,
    mode: str = "outer",
) -> np.ndarray:
    cell_bin = (cell_masks == cid)
    outline = find_boundaries(cell_bin, mode=mode)
    if thickness and thickness > 1:
        outline = morphology.dilation(outline, footprint=morphology.disk(int(thickness)))
    return outline


def overlay_outline_yellow(gray_u8: np.ndarray, outline: np.ndarray) -> np.ndarray:
    rgb = np.stack([gray_u8, gray_u8, gray_u8], axis=-1).copy()
    rgb[outline] = np.array([255, 255, 0], dtype=np.uint8)
    return rgb


def save_cell_pngs_for_image(
    img: np.ndarray,
    cell_masks: np.ndarray,
    cell_geom_df: pd.DataFrame,
    ch_index: dict,
    out_base: Path,
    filename_stem: str,
    pad: int = 8,
    min_cell_area: int = 0,
    bbox_on_crop_edge: bool = True,
    write_outlines: bool = True,
    outline_thickness: int = 2,
):
    out_base.mkdir(parents=True, exist_ok=True)

    dapi = img[ch_index["DAPI"]]
    Phalloidin = img[ch_index["Phalloidin"]]
    lamp1 = img[ch_index["LAMP1"]]
    prab = img[ch_index["pRAB10"]]

    H, W = dapi.shape

    dapi_u8 = to_uint8(dapi)
    Phalloidin_u8 = to_uint8(Phalloidin)
    lamp1_u8 = to_uint8(lamp1)
    prab_u8 = to_uint8(prab)

    for _, r in cell_geom_df.iterrows():
        cid = int(r["cell_id"])
        area = int(r["cell_area_px_geom"])
        if area < min_cell_area:
            continue

        y0, x0, y1, x1 = square_crop_coords(
            r["bbox_ymin"], r["bbox_xmin"], r["bbox_ymax"], r["bbox_xmax"],
            H=H, W=W, pad=pad
        )

        cell_dir = out_base / f"{filename_stem}_cell{cid:04d}"
        cell_dir.mkdir(parents=True, exist_ok=True)

        dapi_crop = dapi_u8[y0:y1, x0:x1]
        Phalloidin_crop = Phalloidin_u8[y0:y1, x0:x1]
        lamp1_crop = lamp1_u8[y0:y1, x0:x1]
        prab_crop = prab_u8[y0:y1, x0:x1]

        _imsave(cell_dir / "DAPI_gray.png",  dapi_crop)
        _imsave(cell_dir / "Phalloidin_gray.png",  Phalloidin_crop)
        _imsave(cell_dir / "LAMP1_gray.png", lamp1_crop)
        _imsave(cell_dir / "pRAB10_gray.png", prab_crop)

        merged = make_merged_rgb(dapi_crop, Phalloidin_crop, lamp1_crop, prab_crop)
        _imsave(cell_dir / "merged_color.png", merged)

        if write_outlines:
            outline_crop = cellpose_outline_from_labels(
                cell_masks=cell_masks[y0:y1, x0:x1],
                cid=cid,
                thickness=outline_thickness,
                mode="outer",
            )

            _imsave(cell_dir / "DAPI_outline_yellow.png",  overlay_outline_yellow(dapi_crop, outline_crop))
            _imsave(cell_dir / "Phalloidin_outline_yellow.png",  overlay_outline_yellow(Phalloidin_crop, outline_crop))
            _imsave(cell_dir / "LAMP1_outline_yellow.png", overlay_outline_yellow(lamp1_crop, outline_crop))
            _imsave(cell_dir / "pRAB10_outline_yellow.png", overlay_outline_yellow(prab_crop, outline_crop))

            merged_outline = merged.copy()
            merged_outline[outline_crop] = np.array([255, 255, 0], dtype=np.uint8)
            _imsave(cell_dir / "merged_color_outline_yellow.png", merged_outline)

        overlay = merged.copy()

        if bbox_on_crop_edge:
            rr, cc = rectangle_perimeter(
                start=(0, 0),
                end=(overlay.shape[0] - 1, overlay.shape[1] - 1),
                shape=overlay.shape[:2],
            )
        else:
            by0 = int(r["bbox_ymin"]) - y0
            bx0 = int(r["bbox_xmin"]) - x0
            by1 = int(r["bbox_ymax"]) - y0 - 1
            bx1 = int(r["bbox_xmax"]) - x0 - 1
            by0 = max(0, min(by0, overlay.shape[0] - 1))
            bx0 = max(0, min(bx0, overlay.shape[1] - 1))
            by1 = max(0, min(by1, overlay.shape[0] - 1))
            bx1 = max(0, min(bx1, overlay.shape[1] - 1))

            rr, cc = rectangle_perimeter(
                start=(by0, bx0),
                end=(by1, bx1),
                shape=overlay.shape[:2],
            )

        overlay[rr, cc] = 255
        _imsave(cell_dir / "merged_color_bbox.png", overlay)


def save_cell_mask_pngs_for_image(
    cell_masks: np.ndarray,
    cell_geom_df: pd.DataFrame,
    lys_mask: np.ndarray,
    prab_mask: np.ndarray,
    out_base: Path,
    filename_stem: str,
    pad: int = 8,
    min_cell_area: int = 0,
    restrict_to_cell: bool = True,
):
    out_base.mkdir(parents=True, exist_ok=True)
    H, W = cell_masks.shape

    for _, r in cell_geom_df.iterrows():
        cid = int(r["cell_id"])
        area = int(r["cell_area_px_geom"])
        if area < min_cell_area:
            continue

        y0, x0, y1, x1 = square_crop_coords(
            r["bbox_ymin"], r["bbox_xmin"], r["bbox_ymax"], r["bbox_xmax"],
            H=H, W=W, pad=pad
        )

        cell_dir = out_base / f"{filename_stem}_cell{cid:04d}"
        masks_dir = cell_dir / "masks"
        masks_dir.mkdir(parents=True, exist_ok=True)

        lys_crop = lys_mask[y0:y1, x0:x1]
        prab_crop = prab_mask[y0:y1, x0:x1]

        if restrict_to_cell:
            cell_crop = (cell_masks[y0:y1, x0:x1] == cid)
            lys_crop = lys_crop & cell_crop
            prab_crop = prab_crop & cell_crop

        _imsave(masks_dir / "lys_mask.png",  (lys_crop.astype(np.uint8) * 255))
        _imsave(masks_dir / "prab_mask.png", (prab_crop.astype(np.uint8) * 255))


# ----------------------------
# Structure segmentation
# ----------------------------
@dataclass
class StructureSegParams:
    intensity_scaling_param: Tuple[float, float]
    min_area: int
    blur_sigma: float
    log_sigma_1: float
    log_cutoff_1: float
    log_sigma_2: float
    log_cutoff_2: float
    log_sigma_3: float
    log_cutoff_3: float
    vesselness_sigma: Tuple[float, ...]
    vesselness_cutoff: float


def segment_structures_acis_style(channel_img: np.ndarray, params: StructureSegParams) -> np.ndarray:
    ch = channel_img.astype(float)

    m, s = norm.fit(ch.flatten())
    stretch_min = max(m - params.intensity_scaling_param[0] * s, float(ch.min()))
    stretch_max = min(m + params.intensity_scaling_param[1] * s, float(ch.max()))
    if stretch_max <= stretch_min:
        return np.zeros_like(ch, dtype=bool)

    ch_n = np.clip(ch, stretch_min, stretch_max)
    image_norm = (ch_n - stretch_min) / (stretch_max - stretch_min)

    blurred = filters.gaussian(image_norm, sigma=params.blur_sigma)

    log_1 = -1.0 * (params.log_sigma_1**2) * gaussian_laplace(blurred, sigma=params.log_sigma_1)
    log_2 = -1.0 * (params.log_sigma_2**2) * gaussian_laplace(blurred, sigma=params.log_sigma_2)
    log_3 = -1.0 * (params.log_sigma_3**2) * gaussian_laplace(blurred, sigma=params.log_sigma_3)

    log_mask = (log_1 > params.log_cutoff_1) | (log_2 > params.log_cutoff_2) | (log_3 > params.log_cutoff_3)

    vesselness = filters.frangi(
        blurred, sigmas=list(params.vesselness_sigma), black_ridges=False
    ) > params.vesselness_cutoff

    combined = log_mask | vesselness
    filled = ndi.binary_fill_holes(combined)
    cleaned = morphology.remove_small_objects(filled, min_size=params.min_area)

    return cleaned.astype(bool)


def segment_lysosomes(channel_img: np.ndarray, params: StructureSegParams) -> np.ndarray:
    lys_ch = channel_img.astype(float)

    m, s = norm.fit(lys_ch.flatten())
    stretch_min = max(m - params.intensity_scaling_param[0] * s, float(lys_ch.min()))
    stretch_max = min(m + params.intensity_scaling_param[1] * s, float(lys_ch.max()))
    if stretch_max <= stretch_min:
        return np.zeros_like(lys_ch, dtype=bool)

    lys_ch_n = np.clip(lys_ch, stretch_min, stretch_max)
    image_norm = (lys_ch_n - stretch_min) / (stretch_max - stretch_min)

    blurred = filters.gaussian(image_norm, sigma=params.blur_sigma)

    triangle_cutoff = filters.threshold_triangle(blurred)
    global_median_cutoff = np.percentile(blurred, 50)
    th_low_cutoff = (triangle_cutoff + global_median_cutoff) / 2.0
    img_low_level = blurred > th_low_cutoff

    img_low_level_small = morphology.remove_small_objects(img_low_level, min_size=int(params.min_area), connectivity=1)
    img_low_level_small_grow = morphology.dilation(img_low_level_small, footprint=morphology.disk(2))

    otsu_cutoff = 0.333 * filters.threshold_otsu(blurred)
    img_high_level = np.zeros_like(img_low_level_small_grow, dtype=bool)

    lab_low, num_obj = morphology.label(img_low_level_small_grow, return_num=True, connectivity=1)
    for idx in range(num_obj):
        single_obj = lab_low == (idx + 1)
        if np.count_nonzero(single_obj) == 0:
            continue
        try:
            local_otsu = filters.threshold_otsu(blurred[single_obj])
        except Exception:
            local_otsu = 0.0
        if local_otsu > otsu_cutoff:
            mask_condition = np.logical_and(blurred > 0.98 * local_otsu, single_obj)
            img_high_level[mask_condition] = True

    log_sigma = params.log_sigma_1
    log_response = -1.0 * (log_sigma**2) * gaussian_laplace(blurred, sigma=log_sigma)
    bw_extra = log_response > 0.09
    bw_extra[~img_low_level_small_grow] = False

    bw_final = np.logical_or(bw_extra, img_high_level)

    filled = ndi.binary_fill_holes(bw_final)
    labeled_filled = morphology.label(filled, connectivity=1)
    lysosome_mask = morphology.remove_small_objects(labeled_filled, min_size=int(params.min_area)) > 0

    return lysosome_mask.astype(bool)


# ----------------------------
# Object assignment per cell
# ----------------------------
def label_objects_within_cells(
    cell_masks: np.ndarray,
    obj_mask: np.ndarray,
) -> Tuple[np.ndarray, Dict[int, int]]:
    cell_ids = np.unique(cell_masks)
    cell_ids = cell_ids[cell_ids != 0]

    labels_global = np.zeros_like(obj_mask, dtype=np.int32)
    parent_cell: Dict[int, int] = {}
    current_label = 1

    for cid in cell_ids:
        single_cell_mask = (cell_masks == cid)
        obj_in_cell = obj_mask & single_cell_mask
        labeled_in_cell, n = ndi.label(obj_in_cell)

        if n == 0:
            continue

        for ll in np.unique(labeled_in_cell)[1:]:
            labels_global[labeled_in_cell == ll] = current_label
            parent_cell[current_label] = int(cid)
            current_label += 1

    return labels_global, parent_cell


# ----------------------------
# Colocalization per cell
# ----------------------------
def compute_cell_coloc(
    cell_masks: np.ndarray,
    lys_int: np.ndarray,
    prab_int: np.ndarray,
    min_pixels: int = 50,
) -> pd.DataFrame:
    cell_ids = np.unique(cell_masks)
    cell_ids = cell_ids[cell_ids != 0]

    rows: List[dict] = []

    for cid in cell_ids:
        m = (cell_masks == cid)
        npx = int(m.sum())
        if npx < min_pixels:
            continue

        a = lys_int[m].astype(np.float64)
        b = prab_int[m].astype(np.float64)

        if a.std() == 0 or b.std() == 0:
            pearson_r = np.nan
        else:
            pearson_r = float(np.corrcoef(a, b)[0, 1])

        try:
            tA = float(threshold_otsu(a)) if np.unique(a).size > 1 else 0.0
        except Exception:
            tA = 0.0
        try:
            tB = float(threshold_otsu(b)) if np.unique(b).size > 1 else 0.0
        except Exception:
            tB = 0.0

        a_pos = a > tA
        b_pos = b > tB

        denomA = float(a[a_pos].sum())
        denomB = float(b[b_pos].sum())

        M1 = float(a[a_pos & b_pos].sum() / denomA) if denomA > 0 else np.nan
        M2 = float(b[b_pos & a_pos].sum() / denomB) if denomB > 0 else np.nan

        rows.append({
            "cell_id": int(cid),
            "cell_pixels": npx,
            "pearson_r": pearson_r,
            "manders_M1_lys_in_prab": M1,
            "manders_M2_prab_in_lys": M2,
            "threshold_cutoff_lys": tA,
            "threshold_cutoff_prab": tB,
        })

    return pd.DataFrame(rows)


def compute_cell_geometry(cell_masks: np.ndarray) -> pd.DataFrame:
    props = regionprops_table(
        cell_masks,
        properties=("label", "area", "centroid", "bbox")
    )
    df = pd.DataFrame(props).rename(columns={
        "label": "cell_id",
        "area": "cell_area_px_geom",
        "centroid-0": "cell_centroid_y",
        "centroid-1": "cell_centroid_x",
        "bbox-0": "bbox_ymin",
        "bbox-1": "bbox_xmin",
        "bbox-2": "bbox_ymax",
        "bbox-3": "bbox_xmax",
    })
    df["cell_id"] = df["cell_id"].astype(int)
    return df


def compute_per_cell_area_metrics(
    cell_masks: np.ndarray,
    nuc_masks: np.ndarray,
    lys_mask: np.ndarray,
    prab_mask: np.ndarray,
) -> pd.DataFrame:
    flat_cells = cell_masks.ravel()
    max_lab = int(flat_cells.max())
    counts = np.bincount(flat_cells, minlength=max_lab + 1)
    nuc_overlap = np.bincount(flat_cells, weights=(nuc_masks.ravel() > 0).astype(int), minlength=max_lab + 1)
    lys_overlap = np.bincount(flat_cells, weights=lys_mask.ravel().astype(int), minlength=max_lab + 1)
    prab_overlap = np.bincount(flat_cells, weights=prab_mask.ravel().astype(int), minlength=max_lab + 1)

    rows = []
    for cid in range(1, max_lab + 1):
        if counts[cid] == 0:
            continue
        rows.append({
            "cell_id": int(cid),
            "cell_area_px": int(counts[cid]),
            "nuc_area_px_in_cell": int(nuc_overlap[cid]),
            "lys_segmented_area_px_in_cell": int(lys_overlap[cid]),
            "prab_segmented_area_px_in_cell": int(prab_overlap[cid]),
        })
    return pd.DataFrame(rows)


# ----------------------------
# Cellpose wrappers
# ----------------------------
@dataclass
class CellposeParams:
    diameter: float = 120
    batch_size: int = 32
    flow_threshold: float = 0.4
    cellprob_threshold: float = 0.0
    tile_norm_blocksize: int = 0


def run_cellpose_cells(
    model: models.CellposeModel,
    dapi: np.ndarray,
    Phalloidin: np.ndarray,
    params: CellposeParams,
) -> np.ndarray:
    stack = np.stack([dapi, Phalloidin], axis=0)
    masks, flows, styles = model.eval(
        stack,
        batch_size=params.batch_size,
        diameter=params.diameter,
        flow_threshold=params.flow_threshold,
        cellprob_threshold=params.cellprob_threshold,
        normalize={"tile_norm_blocksize": params.tile_norm_blocksize},
    )
    return masks


def run_cellpose_nuclei(
    model: models.CellposeModel,
    dapi: np.ndarray,
    params: CellposeParams,
) -> np.ndarray:
    masks, flows, styles = model.eval(
        dapi,
        batch_size=params.batch_size,
        diameter=params.diameter,
        flow_threshold=params.flow_threshold,
        cellprob_threshold=params.cellprob_threshold,
        normalize={"tile_norm_blocksize": params.tile_norm_blocksize},
    )
    return masks


# ----------------------------
# Per-image pipeline config
# ----------------------------
@dataclass
class PipelineConfig:
    n_channels: int = 4
    channel_names: Tuple[str, str, str, str] = ("DAPI", "Phalloidin", "LAMP1", "pRAB10")

    cellpose: CellposeParams = field(default_factory=CellposeParams)

    edge_filter_mode: str = "bbox"
    edge_margin_px: int = 0

    lys_params: StructureSegParams = field(default_factory=lambda: StructureSegParams(
        intensity_scaling_param=(3, 19),
        min_area=5,
        blur_sigma=1,
        log_sigma_1=3, log_cutoff_1=0.13,
        log_sigma_2=2, log_cutoff_2=0.08,
        log_sigma_3=1, log_cutoff_3=0.06,
        vesselness_sigma=(1,),
        vesselness_cutoff=0.3,
    ))

    prab_params: StructureSegParams = field(default_factory=lambda: StructureSegParams(
        intensity_scaling_param=(4, 9),
        min_area=5,
        blur_sigma=2,
        log_sigma_1=3, log_cutoff_1=0.12,
        log_sigma_2=2, log_cutoff_2=0.10,
        log_sigma_3=1, log_cutoff_3=0.08,
        vesselness_sigma=(1,),
        vesselness_cutoff=0.5,
    ))

    coloc_min_pixels: int = 50

    write_cell_pngs: bool = True
    cell_png_pad: int = 8
    cell_png_min_area: int = 0
    bbox_on_crop_edge: bool = True

    write_outlines: bool = True
    outline_thickness: int = 2

    write_mask_pngs: bool = True
    restrict_mask_to_cell: bool = True


# ----------------------------
# Per-cell spatial metrics
# ----------------------------
def compute_per_cell_signal_spatial_metrics(
    cell_masks: np.ndarray,
    nuc_masks: np.ndarray,
    signal_mask: np.ndarray,
    signal_int: np.ndarray,
    prefix: str,
    perinuclear_radius_px: float = 10.0,
) -> pd.DataFrame:
    cell_ids = np.unique(cell_masks)
    cell_ids = cell_ids[cell_ids != 0]

    rows = []
    r = float(perinuclear_radius_px)

    for cid in cell_ids:
        cell_m = (cell_masks == cid)

        nuc_m = (nuc_masks > 0) & cell_m
        if nuc_m.sum() == 0:
            rows.append({
                "cell_id": int(cid),
                "nuc_centroid_y": np.nan,
                "nuc_centroid_x": np.nan,
                f"{prefix}_centroid_y": np.nan,
                f"{prefix}_centroid_x": np.nan,
                f"dist_nuc_to_{prefix}_centroid_px": np.nan,
                f"{prefix}_dist_mean_px": np.nan,
                f"{prefix}_dist_median_px": np.nan,
                f"{prefix}_dist_p90_px": np.nan,
                f"{prefix}_perinuclear_frac_r{int(r)}px": np.nan,
                f"{prefix}_pixels_in_cell": int((signal_mask & cell_m).sum()),
            })
            continue

        nuc_yx = np.argwhere(nuc_m)
        nuc_cy, nuc_cx = nuc_yx.mean(axis=0)

        sig_m = signal_mask & cell_m
        n_sig_px = int(sig_m.sum())
        if n_sig_px == 0:
            rows.append({
                "cell_id": int(cid),
                "nuc_centroid_y": float(nuc_cy),
                "nuc_centroid_x": float(nuc_cx),
                f"{prefix}_centroid_y": np.nan,
                f"{prefix}_centroid_x": np.nan,
                f"dist_nuc_to_{prefix}_centroid_px": np.nan,
                f"{prefix}_dist_mean_px": np.nan,
                f"{prefix}_dist_median_px": np.nan,
                f"{prefix}_dist_p90_px": np.nan,
                f"{prefix}_perinuclear_frac_r{int(r)}px": 0.0,
                f"{prefix}_pixels_in_cell": 0,
            })
            continue

        sig_coords = np.argwhere(sig_m)
        weights = signal_int[sig_m].astype(np.float64)
        wsum = weights.sum()

        if wsum > 0:
            sig_cy = float((sig_coords[:, 0] * weights).sum() / wsum)
            sig_cx = float((sig_coords[:, 1] * weights).sum() / wsum)
        else:
            sig_cy, sig_cx = sig_coords.mean(axis=0).astype(float)

        d_centroid = float(np.hypot(sig_cy - nuc_cy, sig_cx - nuc_cx))

        dy = sig_coords[:, 0].astype(np.float64) - nuc_cy
        dx = sig_coords[:, 1].astype(np.float64) - nuc_cx
        dists = np.hypot(dy, dx)

        perinu_frac = float((dists <= r).mean()) if dists.size else np.nan

        rows.append({
            "cell_id": int(cid),
            "nuc_centroid_y": float(nuc_cy),
            "nuc_centroid_x": float(nuc_cx),
            f"{prefix}_centroid_y": float(sig_cy),
            f"{prefix}_centroid_x": float(sig_cx),
            f"dist_nuc_to_{prefix}_centroid_px": d_centroid,
            f"{prefix}_dist_mean_px": float(dists.mean()),
            f"{prefix}_dist_median_px": float(np.median(dists)),
            f"{prefix}_dist_p90_px": float(np.percentile(dists, 90)),
            f"{prefix}_perinuclear_frac_r{int(r)}px": perinu_frac,
            f"{prefix}_pixels_in_cell": n_sig_px,
        })

    return pd.DataFrame(rows)

# ----------------------------
# Per-image processing
# ----------------------------
def process_one_image(
    row: pd.Series,
    cfg: PipelineConfig,
    cellpose_model: models.CellposeModel,
    out_dir: Optional[Path] = None,
) -> Dict[str, pd.DataFrame]:

    def safe_merge_on_id(left: pd.DataFrame, right: pd.DataFrame, on: str = "cell_id") -> pd.DataFrame:
        if right is None or right.empty:
            return left.copy()
        left_cols = set(left.columns)
        right_cols = [c for c in right.columns if c != on]
        to_take = [c for c in right_cols if c not in left_cols]
        if not to_take:
            return left.copy()
        right_sel = right[[on] + to_take].copy()
        return left.merge(right_sel, on=on, how="left", validate="one_to_one")

    img_path = row.get("image_path", None)
    if img_path is None or str(img_path) == "nan":
        raise FileNotFoundError(f"No image_path for filename={row.get('filename')}")

    img_path = Path(img_path)

    # --- ND2 load + max-projection over Z → (C, H, W) ---
    img = load_nd2_as_czyx(img_path)

    ch_index = get_channel_indices(row, n_channels=cfg.n_channels)
    for chname in cfg.channel_names:
        if chname not in ch_index:
            raise KeyError(f"Missing channel '{chname}' for image {img_path.name}. Got: {list(ch_index.keys())}")

    dapi = img[ch_index["DAPI"]]
    Phalloidin = img[ch_index["Phalloidin"]]
    lys  = img[ch_index["LAMP1"]]
    prab = img[ch_index["pRAB10"]]

    cell_masks = run_cellpose_cells(cellpose_model, dapi, Phalloidin, cfg.cellpose)
    nuc_masks  = run_cellpose_nuclei(cellpose_model, dapi, cfg.cellpose)

    cell_geom_df = compute_cell_geometry(cell_masks)
    cell_geom_df.insert(0, "filename", img_path.name)

    filename_stem = img_path.stem

    lys_mask  = segment_lysosomes(lys, cfg.lys_params)
    prab_mask = segment_structures_acis_style(prab, cfg.prab_params)

    per_cell_area_df = compute_per_cell_area_metrics(
        cell_masks=cell_masks,
        nuc_masks=nuc_masks,
        lys_mask=lys_mask,
        prab_mask=prab_mask,
    )

    edge_margin_px    = getattr(cfg, "edge_margin_px", 0)
    overlap_thresh    = getattr(cfg, "nuc_overlap_thresh", 0.9)
    area_ratio_thresh = getattr(cfg, "nuc_area_ratio_thresh", 1.25)

    edge_flags = compute_edge_flags_bbox(cell_masks, margin=edge_margin_px)
    overlap_frac_dict, area_ratio_dict = compute_nuc_overlap_and_area_ratio(cell_masks, nuc_masks)

    if not per_cell_area_df.empty:
        per_cell_area_df["flag_edge"] = per_cell_area_df["cell_id"].map(edge_flags).fillna(False).astype(bool)
        per_cell_area_df["nuc_overlap_frac"] = per_cell_area_df["cell_id"].map(overlap_frac_dict).fillna(0.0).astype(float)
        per_cell_area_df["cell_to_nuc_area_ratio"] = per_cell_area_df["cell_id"].map(area_ratio_dict).fillna(np.inf).astype(float)
        per_cell_area_df["flag_nucleus_only"] = (
            (per_cell_area_df["nuc_overlap_frac"] >= overlap_thresh)
            & (per_cell_area_df["cell_to_nuc_area_ratio"] <= area_ratio_thresh)
        )

    _lys_r       = lys_mask.ravel().astype(int)
    _prab_r      = prab_mask.ravel().astype(int)
    _lys_int_r   = lys.ravel().astype(np.float64)
    _prab_int_r  = prab.ravel().astype(np.float64)
    _flat_cells  = cell_masks.ravel()
    _max_cell    = int(_flat_cells.max())

    _cell_area_bc    = np.bincount(_flat_cells, minlength=_max_cell + 1)
    _overlap_px_bc   = np.bincount(_flat_cells, weights=_lys_r * _prab_r,              minlength=_max_cell + 1)
    _lys_px_bc       = np.bincount(_flat_cells, weights=_lys_r,                         minlength=_max_cell + 1)
    _prab_px_bc      = np.bincount(_flat_cells, weights=_prab_r,                        minlength=_max_cell + 1)
    _prab_in_lys_bc  = np.bincount(_flat_cells, weights=_prab_int_r * _lys_r,          minlength=_max_cell + 1)
    _prab_out_lys_bc = np.bincount(_flat_cells, weights=_prab_int_r * (1 - _lys_r),    minlength=_max_cell + 1)
    _lys_in_prab_bc  = np.bincount(_flat_cells, weights=_lys_int_r  * _prab_r,         minlength=_max_cell + 1)
    _lys_out_prab_bc = np.bincount(_flat_cells, weights=_lys_int_r  * (1 - _prab_r),   minlength=_max_cell + 1)

    _overlap_rows = []
    for _cid in range(1, _max_cell + 1):
        if _cell_area_bc[_cid] == 0:
            continue
        _n_lys      = _lys_px_bc[_cid]
        _n_prab     = _prab_px_bc[_cid]
        _n_out_lys  = _cell_area_bc[_cid] - _n_lys
        _n_out_prab = _cell_area_bc[_cid] - _n_prab

        _mean_prab_in  = _prab_in_lys_bc[_cid]  / _n_lys      if _n_lys      > 0 else np.nan
        _mean_prab_out = _prab_out_lys_bc[_cid] / _n_out_lys  if _n_out_lys  > 0 else np.nan
        _mean_lys_in   = _lys_in_prab_bc[_cid]  / _n_prab     if _n_prab     > 0 else np.nan
        _mean_lys_out  = _lys_out_prab_bc[_cid] / _n_out_prab if _n_out_prab > 0 else np.nan

        _overlap_rows.append({
            "cell_id":                    int(_cid),
            "mask_overlap_px":            int(_overlap_px_bc[_cid]),
            "mean_prab_int_in_lys_mask":  _mean_prab_in,
            "mean_prab_int_out_lys_mask": _mean_prab_out,
            "mean_lys_int_in_prab_mask":  _mean_lys_in,
            "mean_lys_int_out_lys_mask": _mean_lys_out,
            "prab_enrichment_in_lys": _mean_prab_in  / _mean_prab_out if (_mean_prab_out and _mean_prab_out > 0 and not np.isnan(_mean_prab_in))  else np.nan,
            "lys_enrichment_in_prab": _mean_lys_in   / _mean_lys_out  if (_mean_lys_out  and _mean_lys_out  > 0 and not np.isnan(_mean_lys_in))   else np.nan,
        })

    mask_overlap_df = pd.DataFrame(_overlap_rows)

    lys_labels_global, lys_parent_cell   = label_objects_within_cells(cell_masks, lys_mask)
    prab_labels_global, prab_parent_cell = label_objects_within_cells(cell_masks, prab_mask)

    lys_props = regionprops_table(
        lys_labels_global,
        intensity_image=lys,
        properties=("label", "area", "min_intensity", "mean_intensity", "max_intensity"),
    )
    lys_df = pd.DataFrame(lys_props)
    if not lys_df.empty:
        lys_df["cell_id"] = lys_df["label"].map(lys_parent_cell)
        lys_df.insert(0, "filename", img_path.name)

    prab_props = regionprops_table(
        prab_labels_global,
        intensity_image=prab,
        properties=("label", "area", "min_intensity", "mean_intensity", "max_intensity"),
    )
    prab_df = pd.DataFrame(prab_props)
    if not prab_df.empty:
        prab_df["cell_id"] = prab_df["label"].map(prab_parent_cell)
        prab_df.insert(0, "filename", img_path.name)

    cell_coloc_df = compute_cell_coloc(cell_masks, lys, prab, min_pixels=cfg.coloc_min_pixels)
    if not cell_coloc_df.empty:
        cell_coloc_df.insert(0, "filename", img_path.name)

    lys_spatial_df = compute_per_cell_signal_spatial_metrics(
        cell_masks=cell_masks,
        nuc_masks=nuc_masks,
        signal_mask=lys_mask,
        signal_int=lys,
        prefix="lys",
        perinuclear_radius_px=10.0,
    )

    prab_spatial_df = compute_per_cell_signal_spatial_metrics(
        cell_masks=cell_masks,
        nuc_masks=nuc_masks,
        signal_mask=prab_mask,
        signal_int=prab,
        prefix="prab",
        perinuclear_radius_px=10.0,
    )

    if not cell_geom_df.empty:
        cell_df = cell_geom_df.copy()
    elif not cell_coloc_df.empty:
        cell_df = cell_coloc_df.copy()
    else:
        cell_df = pd.DataFrame(columns=["filename", "cell_id"])

    if "filename" not in cell_df.columns:
        cell_df.insert(0, "filename", img_path.name)

    if not per_cell_area_df.empty:
        cell_df = safe_merge_on_id(cell_df, per_cell_area_df, on="cell_id")

    if not cell_coloc_df.empty:
        geometry_cols = {
            "filename", "cell_id",
            "bbox_ymin", "bbox_xmin", "bbox_ymax", "bbox_xmax",
            "cell_centroid_y", "cell_centroid_x", "cell_area_px_geom"
        }
        coloc_cols = [c for c in cell_coloc_df.columns if c not in geometry_cols]
        if coloc_cols:
            coloc_sel = cell_coloc_df[["cell_id"] + coloc_cols].copy()
            cell_df = safe_merge_on_id(cell_df, coloc_sel, on="cell_id")

    if not lys_spatial_df.empty:
        cell_df = safe_merge_on_id(cell_df, lys_spatial_df, on="cell_id")

    if not prab_spatial_df.empty:
        prab_sel = prab_spatial_df.drop(columns=["nuc_centroid_y", "nuc_centroid_x"], errors="ignore")
        cell_df = safe_merge_on_id(cell_df, prab_sel, on="cell_id")

    if not mask_overlap_df.empty:
        cell_df = safe_merge_on_id(cell_df, mask_overlap_df, on="cell_id")

    if not lys_df.empty:
        lys_agg = lys_df.groupby("cell_id").agg(
            lys_obj_count=("label", "count"),
            lys_obj_area_sum_px=("area", "sum"),
            lys_obj_area_mean_px=("area", "mean"),
        ).reset_index()
        cell_df = safe_merge_on_id(cell_df, lys_agg, on="cell_id")
    else:
        cell_df["lys_obj_count"]        = cell_df.get("lys_obj_count", 0)
        cell_df["lys_obj_area_sum_px"]  = cell_df.get("lys_obj_area_sum_px", 0.0)
        cell_df["lys_obj_area_mean_px"] = cell_df.get("lys_obj_area_mean_px", 0.0)

    if not prab_df.empty:
        prab_agg = prab_df.groupby("cell_id").agg(
            prab_obj_count=("label", "count"),
            prab_obj_area_sum_px=("area", "sum"),
            prab_obj_area_mean_px=("area", "mean"),
        ).reset_index()
        cell_df = safe_merge_on_id(cell_df, prab_agg, on="cell_id")
    else:
        cell_df["prab_obj_count"]        = cell_df.get("prab_obj_count", 0)
        cell_df["prab_obj_area_sum_px"]  = cell_df.get("prab_obj_area_sum_px", 0.0)
        cell_df["prab_obj_area_mean_px"] = cell_df.get("prab_obj_area_mean_px", 0.0)

    if not cell_df.empty:
        for c in ["lys_obj_count", "prab_obj_count"]:
            if c in cell_df.columns:
                cell_df[c] = cell_df[c].fillna(0).astype(int)
        for c in ["lys_obj_area_sum_px", "prab_obj_area_sum_px", "lys_obj_area_mean_px", "prab_obj_area_mean_px"]:
            if c in cell_df.columns:
                cell_df[c] = cell_df[c].fillna(0.0)

    flag_cols = ["flag_edge", "nuc_overlap_frac", "cell_to_nuc_area_ratio", "flag_nucleus_only"]
    for fc in flag_cols:
        if fc not in cell_df.columns:
            cell_df[fc] = np.nan if ("frac" in fc or "ratio" in fc) else False

    cols = list(cell_df.columns)
    if "filename" in cols and "cell_id" in cols:
        front = ["filename", "cell_id"]
        rest  = [c for c in cols if c not in front]
        cell_df = cell_df[front + rest]

    if cfg.write_cell_pngs and out_dir is not None:
        save_cell_pngs_for_image(
            img=img,
            cell_masks=cell_masks,
            cell_geom_df=cell_df,
            ch_index=ch_index,
            out_base=out_dir / "cell_pngs" / filename_stem,
            filename_stem=filename_stem,
            pad=cfg.cell_png_pad,
            min_cell_area=cfg.cell_png_min_area,
            bbox_on_crop_edge=cfg.bbox_on_crop_edge,
            write_outlines=cfg.write_outlines,
            outline_thickness=cfg.outline_thickness,
        )

    if cfg.write_mask_pngs and out_dir is not None:
        save_cell_mask_pngs_for_image(
            cell_masks=cell_masks,
            cell_geom_df=cell_df,
            lys_mask=lys_mask,
            prab_mask=prab_mask,
            out_base=out_dir / "cell_pngs" / filename_stem,
            filename_stem=filename_stem,
            pad=cfg.cell_png_pad,
            min_cell_area=cfg.cell_png_min_area,
            restrict_to_cell=cfg.restrict_mask_to_cell,
        )

    return {
        "cell_df": cell_df,
        "lys_df":  lys_df,
        "prab_df": prab_df,
    }

# ----------------------------
# Batch runner
# ----------------------------
def run_on_folder(
    samplesheet_csv: Path,
    img_dir: Path,
    out_dir: Path,
    pattern: str = "*.nd2",       # changed from *.tif
    use_gpu: bool = True,
    cfg: Optional[PipelineConfig] = None,
) -> Dict[str, Path]:
    cfg = cfg or PipelineConfig()
    out_dir.mkdir(parents=True, exist_ok=True)

    samplesheet = pd.read_csv(samplesheet_csv)

    img_lookup = build_img_lookup(img_dir, pattern=pattern)
    samplesheet = add_image_paths(samplesheet, img_lookup)

    present = samplesheet["image_path"].notna()
    missing = samplesheet.loc[~present, "filename"].tolist()
    if missing:
        print(f"[WARN] {len(missing)} filenames not found in folder; skipping first few: {missing[:5]}")
    samplesheet = samplesheet.loc[present].reset_index(drop=True)

    cellpose_model = models.CellposeModel(gpu=use_gpu)

    cells_all = []
    lys_all = []
    prab_all = []
    failures = []

    for i, row in samplesheet.iterrows():
        try:
            res = process_one_image(row, cfg, cellpose_model, out_dir=out_dir)

            cell_df = res.get("cell_df")
            if cell_df is not None and not cell_df.empty:
                cells_all.append(cell_df)

            if res.get("lys_df") is not None and not res["lys_df"].empty:
                lys_all.append(res["lys_df"])
            if res.get("prab_df") is not None and not res["prab_df"].empty:
                prab_all.append(res["prab_df"])

        except Exception as e:
            fname = str(row.get("filename", "UNKNOWN"))
            tb = traceback.format_exc()
            failures.append((fname, type(e).__name__, str(e), tb))
            print(f"[FAIL] {fname}: {type(e).__name__}: {e}")
            continue

        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{len(samplesheet)} images so far")

    out_paths: Dict[str, Path] = {}

    cells_all_df = pd.concat(cells_all, ignore_index=True) if cells_all else pd.DataFrame()
    lys_all_df   = pd.concat(lys_all,   ignore_index=True) if lys_all   else pd.DataFrame()
    prab_all_df  = pd.concat(prab_all,  ignore_index=True) if prab_all  else pd.DataFrame()

    out_paths["cells_all"] = out_dir / "cells_all.csv"
    out_paths["lys_objects_all"] = out_dir / "lys_objects_all.csv"
    out_paths["prab_objects_all"] = out_dir / "prab_objects_all.csv"

    cells_all_df.to_csv(out_paths["cells_all"], index=False)
    lys_all_df.to_csv(out_paths["lys_objects_all"], index=False)
    prab_all_df.to_csv(out_paths["prab_objects_all"], index=False)

    if failures:
        fail_path = out_dir / "failures.csv"
        pd.DataFrame(failures, columns=["filename", "error_type", "error_message", "traceback"]).to_csv(
            fail_path, index=False
        )
        out_paths["failures"] = fail_path
        print(f"[WARN] {len(failures)} failures written to {fail_path}")

    print("Done.")
    return out_paths

In [4]:
if __name__ == "__main__":
    samplesheet_csv = Path("/Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Data/20260706/20260623_WT_A18_iMG/20260717_WT_iMG_samplesheet.csv")
    img_dir = Path("/Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Data/20260706/20260623_WT_A18_iMG/images")
    out_dir = img_dir / "pipeline_outputs_20260720"

    cfg = PipelineConfig(
        write_cell_pngs=True,
        write_outlines=True,
        outline_thickness=2,
        write_mask_pngs=True,
        restrict_mask_to_cell=True,
        cell_png_pad=8,
        cell_png_min_area=0,
        bbox_on_crop_edge=True,
    )

    paths = run_on_folder(samplesheet_csv, img_dir, out_dir, pattern="*.nd2", use_gpu=True, cfg=cfg)
    for k, v in paths.items():
        print(k, "->", v)


Processed 10/77 images so far
Processed 20/77 images so far
Processed 30/77 images so far
Processed 40/77 images so far
Processed 50/77 images so far
Processed 60/77 images so far
Processed 70/77 images so far
Done.
cells_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Data/20260706/20260623_WT_A18_iMG/images/pipeline_outputs_20260720/cells_all.csv
lys_objects_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Data/20260706/20260623_WT_A18_iMG/images/pipeline_outputs_20260720/lys_objects_all.csv
prab_objects_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Data/20260706/20260623_WT_A18_iMG/images/pipeline_outputs_20260720/prab_objects_all.csv


In [3]:
# QC - built this for tiff, need to optimize for multichannel ND2 files

from pathlib import Path
import numpy as np
import pandas as pd
import tifffile
import napari
import nd2

samplesheet_csv = Path("/Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Data/20260507/20260511_SampleSheet.csv")
img_dir = Path("/Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Data/20260507/20260430_endogenousWTmAC")
out_dir = img_dir / "pipeline_outputs_20260515"

# load one matched row
samplesheet = pd.read_csv(samplesheet_csv)
img_lookup = build_img_lookup(img_dir, pattern="*.tif")
# img_lookup = build_img_lookup(img_dir, pattern="*.nd2")
samplesheet = add_image_paths(samplesheet, img_lookup)
row = samplesheet[samplesheet["image_path"].notna()].iloc[0].copy()

cfg = PipelineConfig(
    write_cell_pngs=False,
    write_mask_pngs=False,
    write_outlines=False,
)

model = models.CellposeModel(gpu=True)

# run your pipeline once
res = process_one_image(row, cfg, model, out_dir=None)
cell_df = res["cell_df"]

# reload the image and recompute masks for viewing
# img = nd2.imread(row["image_path"])
img = tifffile.imread(row["image_path"])
ch_index = get_channel_indices(row, n_channels=cfg.n_channels)

dapi = img[ch_index["DAPI"]]
Phalloidin = img[ch_index["Phalloidin"]]
lamp1 = img[ch_index["LAMP1"]]
prab = img[ch_index["pRAB10"]]

cell_masks = run_cellpose_cells(model, dapi, Phalloidin, cfg.cellpose)
nuc_masks = run_cellpose_nuclei(model, dapi, cfg.cellpose)

lys_mask  = segment_lysosomes(lamp1, cfg.lys_params)
prab_mask = segment_structures_acis_style(prab, cfg.prab_params)

qc = cell_df[[
    "cell_id",
    "flag_edge",
    "nuc_overlap_frac",
    "cell_to_nuc_area_ratio",
    "flag_nucleus_only",
    "cell_area_px",
    "nuc_area_px_in_cell",
]].copy()

# define the exact removal rule you want to inspect
qc["flag_removed"] = qc["flag_edge"] | qc["flag_nucleus_only"]

display(
    qc.sort_values(["flag_removed", "flag_edge", "flag_nucleus_only"], ascending=False)
)

def rect_from_bbox(ymin, xmin, ymax, xmax):
    return np.array([
        [ymin, xmin],
        [ymin, xmax],
        [ymax, xmax],
        [ymax, xmin],
    ])

viewer = napari.Viewer()

viewer.add_image(dapi,  name="DAPI",   colormap="blue")
viewer.add_image(Phalloidin,  name="Phalloidin",   colormap="green")
viewer.add_image(lamp1, name="LAMP1",  colormap="red")
viewer.add_image(prab,  name="pRAB10", colormap="magenta")

viewer.add_labels(cell_masks.astype(np.int32), name="cell_masks")
viewer.add_labels(nuc_masks.astype(np.int32),  name="nuc_masks")

lys_mask  = segment_lysosomes(lamp1, cfg.lys_params)
prab_mask = segment_structures_acis_style(prab, cfg.prab_params)

viewer.add_image(lys_mask.astype(np.uint8) * 255,  name="lys_mask")
viewer.add_image(prab_mask.astype(np.uint8) * 255, name="prab_mask")

kept = qc[~qc["flag_removed"]]
removed = qc[qc["flag_removed"]]

kept_shapes = [
    rect_from_bbox(
        cell_df.loc[cell_df["cell_id"] == cid, "bbox_ymin"].iloc[0],
        cell_df.loc[cell_df["cell_id"] == cid, "bbox_xmin"].iloc[0],
        cell_df.loc[cell_df["cell_id"] == cid, "bbox_ymax"].iloc[0],
        cell_df.loc[cell_df["cell_id"] == cid, "bbox_xmax"].iloc[0],
    )
    for cid in kept["cell_id"].astype(int)
]

removed_shapes = [
    rect_from_bbox(
        cell_df.loc[cell_df["cell_id"] == cid, "bbox_ymin"].iloc[0],
        cell_df.loc[cell_df["cell_id"] == cid, "bbox_xmin"].iloc[0],
        cell_df.loc[cell_df["cell_id"] == cid, "bbox_ymax"].iloc[0],
        cell_df.loc[cell_df["cell_id"] == cid, "bbox_xmax"].iloc[0],
    )
    for cid in removed["cell_id"].astype(int)
]

viewer.add_shapes(
    kept_shapes,
    shape_type="rectangle",
    name="kept_cells",
    edge_color="green",
    face_color="transparent",
    edge_width=2,
)

viewer.add_shapes(
    removed_shapes,
    shape_type="rectangle",
    name="removed_cells",
    edge_color="red",
    face_color="transparent",
    edge_width=3,
)

viewer

IndexError: single positional indexer is out-of-bounds